# ML-05 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmedosrf/flyrank-ml-internship-ahmedosrf/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This notebook freezes a transparent Lane 2 baseline for content-refresh prioritization. The score uses only snapshot signals that are available before a review decision. The decline label is used only after ranking, for an honest evaluation receipt; it is never an input to the score.

## 1. My rule and its reason codes

**Rule in plain words.** Prioritize pages that have enough search visibility, appear on page one or two, and have a low observed CTR. Among those candidates, rank higher-volume pages first because a snippet or intent mismatch has more potential reach. This is a review queue, not an automatic rewrite.

**Signal check A — volume / visibility.** This is linked to FlyRank's quick-win logic: a page needs meaningful impressions before an intervention is worth a human's time. The verdict below is **CONFIRMED** when higher-volume buckets show a higher observed decline rate than the low-volume bucket.

**Signal check B — CTR versus position.** This is linked to FlyRank's CTR-fix logic: a page with enough impressions, a position between 1 and 20, and CTR below 0.5% is a plausible snippet or intent-review candidate. The verdict below is **CONFIRMED** when the low-CTR bucket has the highest observed decline rate. Both verdicts are directional associations, not causal claims.

**Reason codes.** `visible_low_ctr_page` means the page passed all three human-readable conditions. `not_prioritized` is retained for the rest of the queue so every row has an explicit reason. The action labels are `review_title_meta` for the positive queue and `monitor` otherwise.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

DATA = Path('../..') / 'data/raw/content_refresh_anonymized.csv'
OUT_DIR = Path('../outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA)
# Evaluation-only receipt. It is never included in SCORE_FEATURES or the CSV queue.
df['is_declining_eval'] = (df['trend_direction'] == 'down').astype(int)

SCORE_FEATURES = ['impressions_90d', 'avg_position', 'ctr']
print('Rows loaded:', len(df))
print('Scoring features:', SCORE_FEATURES)
print('Evaluation base rate (label used after ranking only):', round(df['is_declining_eval'].mean(), 4))

Rows loaded: 30000
Scoring features: ['impressions_90d', 'avg_position', 'ctr']
Evaluation base rate (label used after ranking only): 0.5421


In [2]:
# Signal check A: volume / visibility buckets, with n printed.
df['volume_bucket'] = pd.cut(
    df['impressions_90d'],
    bins=[-1, 499, 2999, float('inf')],
    labels=['<500', '500-2999', '3000+'],
)
volume_table = (
    df.groupby('volume_bucket', observed=False)
      .agg(n=('content_id', 'size'), observed_decline_rate=('is_declining_eval', 'mean'), median_impressions=('impressions_90d', 'median'))
      .reset_index()
)
display(volume_table)
volume_verdict = 'CONFIRMED' if volume_table.loc[volume_table['volume_bucket'] == '500-2999', 'observed_decline_rate'].iloc[0] > volume_table.loc[volume_table['volume_bucket'] == '<500', 'observed_decline_rate'].iloc[0] else 'MIXED'
print('Volume verdict:', volume_verdict)

# Signal check B: CTR buckets among pages with enough volume and a usable page-one/two position.
ctr_position = df[(df['impressions_90d'] >= 500) & (df['avg_position'] > 0) & (df['avg_position'] <= 20)].copy()
ctr_position['ctr_bucket'] = pd.cut(
    ctr_position['ctr'],
    bins=[-float('inf'), 0.5, 1.0, float('inf')],
    labels=['<0.5%', '0.5-1.0%', '>=1.0%'],
)
ctr_table = (
    ctr_position.groupby('ctr_bucket', observed=False)
                .agg(n=('content_id', 'size'), observed_decline_rate=('is_declining_eval', 'mean'), median_position=('avg_position', 'median'))
                .reset_index()
)
display(ctr_table)
ctr_verdict = 'CONFIRMED' if ctr_table.loc[ctr_table['ctr_bucket'] == '<0.5%', 'observed_decline_rate'].iloc[0] > ctr_table['observed_decline_rate'].iloc[1:].max() else 'MIXED'
print('CTR-vs-position verdict:', ctr_verdict)

,volume_bucket,n,observed_decline_rate,median_impressions
0,<500,13274,0.474687,53.0
1,500-2999,8443,0.620632,1230.0
2,3000+,8283,0.569963,8426.0


Volume verdict: CONFIRMED


,ctr_bucket,n,observed_decline_rate,median_position
0,<0.5%,9822,0.626553,8.4
1,0.5-1.0%,1681,0.477097,7.1
2,>=1.0%,520,0.461538,7.5


CTR-vs-position verdict: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

The score is deliberately hand-written: `visible × page_one_two × low_ctr × log1p(impressions)`. It has no fitted weight and no future-window column. The CSV is an output artifact and is intentionally ignored by Git; the notebook regenerates it on every run.

In [3]:
visible = (df['impressions_90d'] >= 500).astype(int)
page_one_two = ((df['avg_position'] > 0) & (df['avg_position'] <= 20)).astype(int)
low_ctr = (df['ctr'] < 0.5).astype(int)

df['score'] = visible * page_one_two * low_ctr * np.log1p(df['impressions_90d'].astype(float))
df['reason_code'] = np.where(df['score'] > 0, 'visible_low_ctr_page', 'not_prioritized')
df['action_label'] = np.where(df['score'] > 0, 'review_title_meta', 'monitor')
queue = (
    df[['content_id','client_id','score','reason_code','action_label','impressions_90d','ctr','avg_position']]
    .drop_duplicates('content_id')
    .sort_values(['score','impressions_90d','content_id'], ascending=[False, False, True])
    .reset_index(drop=True)
)
queue.insert(0, 'rank', np.arange(1, len(queue) + 1))
queue_path = OUT_DIR / 'baseline_action_score.csv'
queue.to_csv(queue_path, index=False)

# Evaluation uses the same ranking, but the evaluation-only label never enters the score.
ranked_eval = queue.merge(df[['content_id','is_declining_eval']], on='content_id', how='left')
def precision_at_k(frame, k):
    return float(frame.head(k)['is_declining_eval'].mean())
metrics = {
    'rows_ranked': int(len(queue)),
    'candidate_rows': int((queue['score'] > 0).sum()),
    'base_rate': float(df['is_declining_eval'].mean()),
    'precision_at_10': precision_at_k(ranked_eval, 10),
    'precision_at_50': precision_at_k(ranked_eval, 50),
    'signal_verdicts': {'volume': volume_verdict, 'ctr_vs_position': ctr_verdict},
    'score_features': SCORE_FEATURES,
}
(OUT_DIR / 'ml05_baseline_metrics.json').write_text(json.dumps(metrics, indent=2) + '\n')
print('Queue written to:', queue_path)
print('Queue rows:', len(queue), '| positive candidates:', int((queue['score'] > 0).sum()))
print('Precision@10:', round(metrics['precision_at_10'], 4), '| Precision@50:', round(metrics['precision_at_50'], 4), '| base rate:', round(metrics['base_rate'], 4))
display(queue.head(10))

Queue written to: ../outputs/baseline_action_score.csv
Queue rows: 30000 | positive candidates: 9759
Precision@10: 0.6 | Precision@50: 0.42 | base rate: 0.5421


,rank,content_id,client_id,score,reason_code,action_label,impressions_90d,ctr,avg_position
0,1,content_5fe46e04994d,client_4e07408562,13.157182,visible_low_ctr_page,review_title_meta,517715,0.14,4.2
1,2,content_aaef01a50def,client_19581e27de,13.156011,visible_low_ctr_page,review_title_meta,517109,0.25,5.4
2,3,content_8c19996aa890,client_4e07408562,13.140700,visible_low_ctr_page,review_title_meta,509252,0.15,2.5
3,4,content_4c36c775b818,client_4e07408562,13.045707,visible_low_ctr_page,review_title_meta,463103,0.41,2.3
4,5,content_1a9e894be2e2,client_19581e27de,12.938876,visible_low_ctr_page,review_title_meta,416180,0.23,4.0
5,6,content_db5989a78dd3,client_4e07408562,12.751624,visible_low_ctr_page,review_title_meta,345111,0.21,5.4
6,7,content_cb112fce36be,client_19581e27de,12.644040,visible_low_ctr_page,review_title_meta,309910,0.16,5.6
7,8,content_36ff89c8214e,client_19581e27de,12.595063,visible_low_ctr_page,review_title_meta,295097,0.05,7.3
8,9,content_8451fc6f034d,client_d029fa3a95,12.514090,visible_low_ctr_page,review_title_meta,272144,0.03,2.3
9,10,content_008fb02c46cb,client_349c41201b,12.374988,visible_low_ctr_page,review_title_meta,236803,0.26,4.4


## 3. Top-10 review

The table below is a manual-style review. Each row states the action, why the rule selected it, and one concrete condition that would make the recommendation wrong. The identifiers are pseudonymous; the queue is for a human analyst to inspect, not a blind automatic edit.

In [4]:
top10 = queue.head(10).copy()
top10['review_line'] = top10.apply(
    lambda r: (
        f"{r['rank']}. action={r['action_label']}; why={r['reason_code']} with {r['impressions_90d']:.0f} impressions, CTR {r['ctr']:.2f}% and position {r['avg_position']:.1f}; "
        f"wrong_if=the low CTR is caused by a measurement anomaly, a one-off SERP change, or the page's intent is not actually eligible for a snippet rewrite."
    ), axis=1
)
for line in top10['review_line']:
    print(line)
print('\nTop-10 table:')
display(top10[['rank','content_id','action_label','reason_code','impressions_90d','ctr','avg_position','review_line']])

1. action=review_title_meta; why=visible_low_ctr_page with 517715 impressions, CTR 0.14% and position 4.2; wrong_if=the low CTR is caused by a measurement anomaly, a one-off SERP change, or the page's intent is not actually eligible for a snippet rewrite.
2. action=review_title_meta; why=visible_low_ctr_page with 517109 impressions, CTR 0.25% and position 5.4; wrong_if=the low CTR is caused by a measurement anomaly, a one-off SERP change, or the page's intent is not actually eligible for a snippet rewrite.
3. action=review_title_meta; why=visible_low_ctr_page with 509252 impressions, CTR 0.15% and position 2.5; wrong_if=the low CTR is caused by a measurement anomaly, a one-off SERP change, or the page's intent is not actually eligible for a snippet rewrite.
4. action=review_title_meta; why=visible_low_ctr_page with 463103 impressions, CTR 0.41% and position 2.3; wrong_if=the low CTR is caused by a measurement anomaly, a one-off SERP change, or the page's intent is not actually eligible

,rank,content_id,action_label,reason_code,impressions_90d,ctr,avg_position,review_line
0,1,content_5fe46e04994d,review_title_meta,visible_low_ctr_page,517715,0.14,4.2,1. action=review_title_meta; why=visible_low_c...
1,2,content_aaef01a50def,review_title_meta,visible_low_ctr_page,517109,0.25,5.4,2. action=review_title_meta; why=visible_low_c...
2,3,content_8c19996aa890,review_title_meta,visible_low_ctr_page,509252,0.15,2.5,3. action=review_title_meta; why=visible_low_c...
3,4,content_4c36c775b818,review_title_meta,visible_low_ctr_page,463103,0.41,2.3,4. action=review_title_meta; why=visible_low_c...
4,5,content_1a9e894be2e2,review_title_meta,visible_low_ctr_page,416180,0.23,4.0,5. action=review_title_meta; why=visible_low_c...
5,6,content_db5989a78dd3,review_title_meta,visible_low_ctr_page,345111,0.21,5.4,6. action=review_title_meta; why=visible_low_c...
6,7,content_cb112fce36be,review_title_meta,visible_low_ctr_page,309910,0.16,5.6,7. action=review_title_meta; why=visible_low_c...
7,8,content_36ff89c8214e,review_title_meta,visible_low_ctr_page,295097,0.05,7.3,8. action=review_title_meta; why=visible_low_c...
8,9,content_8451fc6f034d,review_title_meta,visible_low_ctr_page,272144,0.03,2.3,9. action=review_title_meta; why=visible_low_c...
9,10,content_008fb02c46cb,review_title_meta,visible_low_ctr_page,236803,0.26,4.4,10. action=review_title_meta; why=visible_low_...


## 4. Weak picks + leakage check

The weakest positive picks are pages that pass the rule but have lower volume than the top of the queue. They are useful stress tests: a small absolute change in clicks can move their CTR, so a reviewer should not treat the action as high confidence. The score uses only `impressions_90d`, `avg_position`, and `ctr`; it does not use `trend_direction`, `trend_pct`, any last/previous-window field, or a product decision flag.

In [5]:
weak_positive = queue[queue['score'] > 0].sort_values(['impressions_90d','score']).head(5)
print('Weak positive picks:')
display(weak_positive[['rank','content_id','action_label','reason_code','impressions_90d','ctr','avg_position']])
for forbidden in ['trend_direction', 'trend_pct', 'impressions_last_30d', 'impressions_prev_30d', 'is_declining_eval']:
    assert forbidden not in SCORE_FEATURES, f'Forbidden leakage feature found: {forbidden}'
assert set(SCORE_FEATURES) == {'impressions_90d','avg_position','ctr'}
print('Leakage check passed: no future-window or label-derived input is in SCORE_FEATURES.')

Weak positive picks:


,rank,content_id,action_label,reason_code,impressions_90d,ctr,avg_position
9755,9756,content_4f35dec82501,review_title_meta,visible_low_ctr_page,500,0.2,16.0
9756,9757,content_829eba7c571f,review_title_meta,visible_low_ctr_page,500,0.0,13.8
9757,9758,content_ca56885db527,review_title_meta,visible_low_ctr_page,500,0.4,16.9
9758,9759,content_cd892ad205d3,review_title_meta,visible_low_ctr_page,500,0.0,19.0
9751,9752,content_0184d9cd40b5,review_title_meta,visible_low_ctr_page,501,0.0,17.8


Leakage check passed: no future-window or label-derived input is in SCORE_FEATURES.


## Self-check

- [x] Two signal checks have visible bucket tables with `n`; both are linked to a FlyRank-style flag family.
- [x] Each signal has a one-word verdict: CONFIRMED, OPPOSITE, MIXED, or FALSE.
- [x] One transparent rule produces a score, a reason code, and an action label.
- [x] The ranked queue is written by the notebook to `work/outputs/baseline_action_score.csv`.
- [x] The top ten each have an action, a reason, and a condition that could make the pick wrong.
- [x] The CSV is not committed by design; `work/outputs/ml05_baseline_metrics.json` is the committed receipt.
- [x] No future-window or label-derived column is used as a scoring input.

The lane is now locked as Lane 2: a content-refresh review queue focused on visible pages with a possible CTR/snippet opportunity. This baseline is frozen so a later model has a fair target to beat.